# PAPIT Evaluation Notebook

Runs the full benchmark pipeline described in `EXPERIMENT_PLAN.md`.

| Section | Content | Requires GPU? |
|---|---|---|
| 1 | Dataset preparation (raw files → CSV) | No |
| 2 | LLaVA accuracy benchmark (PAPIT vs Random vs Unpruned) | **Yes** (≥16 GB VRAM) |
| 3 | Paper figures and Table 1 | No |
| 4 | (Optional) Risk awareness ablation | No |

> **Before running Section 2**, download the datasets listed in Section 1 and run Section 1 first.

In [ ]:
import sys
from pathlib import Path
import torch

# Ensure repo root is on path when running from notebooks/
sys.path.insert(0, str(Path().resolve().parent))

DEVICE = (
    "mps"  if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else
    "cpu"
)
print("torch :", torch.__version__)
print("device:", DEVICE)
if DEVICE == "cuda":
    print("VRAM  :", torch.cuda.get_device_properties(0).total_memory // 1024**3, "GB")

---
## Section 1 — Dataset Preparation

Converts raw dataset files into the unified CSV format expected by Section 2.
**Does not require GPU.** Only needs to be run once.

### Download links

| Dataset | Files |
|---|---|
| **GQA** | [images.zip](https://downloads.cs.stanford.edu/nlp/data/gqa/images.zip) · [questions1.2.zip](https://downloads.cs.stanford.edu/nlp/data/gqa/questions1.2.zip) → use `val_balanced_questions.json` |
| **VQA v2** | [val2014 images](http://images.cocodataset.org/zips/val2014.zip) · [questions](https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Val_mscoco.zip) · [annotations](https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Val_mscoco.zip) |
| **TextVQA** | [TextVQA_0.5.1_val.json](https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_val.json) · [train_val_images.zip](https://dl.fbaipublicfiles.com/textvqa/images/train_val_images.zip) |

Extract everything under `data/raw/` before running the cells below.

### 1-A  下載原始資料集

每個資料集只需要下載一次，完成後才執行 1-B 的轉換。
總共約 **33 GB**，建議在下載時先確認磁碟空間。

> 若已下載完畢可直接跳到 **1-B**。

In [ ]:
# 建立資料夾結構（執行一次即可）
!mkdir -p ../data/raw/gqa
!mkdir -p ../data/raw/vqa_v2
!mkdir -p ../data/raw/textvqa

#### GQA（~20 GB images + ~50 MB questions）

問題需要多步推理，例如「這個物品在藍色桌子的左邊還是右邊？」。
對 patch 選擇品質要求高，適合評估 PAPIT 的語意理解能力。

In [ ]:
%%bash
cd ../data/raw/gqa

# 下載圖片與問題檔案
wget -nc https://downloads.cs.stanford.edu/nlp/data/gqa/images.zip
wget -nc https://downloads.cs.stanford.edu/nlp/data/gqa/questions1.2.zip

# 解壓縮
unzip -q images.zip     -d images
unzip -q questions1.2.zip

# 只保留 val 用的問題檔
mv val_balanced_questions.json ./

#### VQA v2（~6 GB images + ~50 MB annotations）

學術界標準 benchmark，結果可直接與其他論文數字對比。
每題有 10 個 human annotation，支援 VQA soft accuracy 評分。

In [ ]:
%%bash
cd ../data/raw/vqa_v2

# 下載圖片
wget -nc http://images.cocodataset.org/zips/val2014.zip
# 下載問題與標注
wget -nc https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Val_mscoco.zip
wget -nc https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Val_mscoco.zip

# 解壓縮
unzip -q val2014.zip                    -d val2014
unzip -q v2_Questions_Val_mscoco.zip    -d .
unzip -q v2_Annotations_Val_mscoco.zip  -d .

#### TextVQA（~7 GB images + ~15 MB annotations）

圖片中含有可讀文字（路標、廣告牌、看板）。
文字通常只佔 1–3 個 patch，但對回答至關重要——是最直接測試 PAPIT prompt-awareness 的 dataset。
Annotation 包含 OCR bounding box，可計算 **patch recall** 指標。

In [ ]:
%%bash
cd ../data/raw/textvqa

# 下載 annotation
wget -nc https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_val.json
# 下載圖片（train + val 共用同一資料夾）
wget -nc https://dl.fbaipublicfiles.com/textvqa/images/train_val_images.zip

# 解壓縮
unzip -q train_val_images.zip -d train_val_images

### 1-B  轉換成 Benchmark CSV

下載完成後，執行以下 cell 將原始檔案轉成統一格式的 CSV。
每個 CSV 欄位：`image_path`, `question`, `answer`, `answer_list`（JSON），TextVQA 另有 `ocr_boxes`。

In [ ]:
# ── Edit these paths to match where you extracted the downloads ──────────────
GQA_QUESTIONS  = "../data/raw/gqa/val_balanced_questions.json"
GQA_IMAGES     = "../data/raw/gqa/images"

VQA_QUESTIONS  = "../data/raw/vqa_v2/v2_OpenEnded_mscoco_val2014_questions.json"
VQA_ANNOT      = "../data/raw/vqa_v2/v2_mscoco_val2014_annotations.json"
VQA_IMAGES     = "../data/raw/vqa_v2/val2014"

TVQA_ANNOT     = "../data/raw/textvqa/TextVQA_0.5.1_val.json"
TVQA_IMAGES    = "../data/raw/textvqa/train_val_images"

# Number of samples per dataset (None = use all)
MAX_GQA   = 1000
MAX_VQA   = 1000
MAX_TVQA  = 500

In [ ]:
from papit.data import load_gqa, load_vqa_v2, load_textvqa
from pathlib import Path

Path("../data").mkdir(exist_ok=True)

gqa_df = load_gqa(GQA_QUESTIONS, GQA_IMAGES, max_samples=MAX_GQA)
gqa_df.to_csv("../data/gqa_val.csv", index=False)
print(f"GQA     : {len(gqa_df):>5} rows → ../data/gqa_val.csv")

vqa_df = load_vqa_v2(VQA_QUESTIONS, VQA_ANNOT, VQA_IMAGES, max_samples=MAX_VQA)
vqa_df.to_csv("../data/vqa_v2_val.csv", index=False)
print(f"VQA v2  : {len(vqa_df):>5} rows → ../data/vqa_v2_val.csv")

tvqa_df = load_textvqa(TVQA_ANNOT, TVQA_IMAGES, max_samples=MAX_TVQA)
tvqa_df.to_csv("../data/textvqa_val.csv", index=False)
print(f"TextVQA : {len(tvqa_df):>5} rows → ../data/textvqa_val.csv")

In [ ]:
# Quick sanity check on each CSV
import pandas as pd, json

for name, path in [("GQA", "../data/gqa_val.csv"),
                   ("VQA v2", "../data/vqa_v2_val.csv"),
                   ("TextVQA", "../data/textvqa_val.csv")]:
    df = pd.read_csv(path)
    sample = df.iloc[0]
    ans_list = json.loads(sample["answer_list"])
    print(f"{name:<8} | {len(df):>5} rows | answer_list len={len(ans_list):>2} | Q: {sample['question'][:60]}")

---
## Section 2 — LLaVA Accuracy Benchmark

Compares three variants at retention ratios {25%, 50%, 75%}:
- **Unpruned** — full LLaVA-1.5 (N = 576 patches)
- **Random** — k patches chosen uniformly at random
- **PAPIT** — top-k by CLIP cross-modal cosine score

**Requires GPU with ≥ 16 GB VRAM.** LLaVA-1.5-7B is loaded once and reused across all samples.

Run each dataset cell independently; results are saved to CSV as each finishes.

In [ ]:
from papit.benchmark.llava_runner import run_llava_benchmark

RETENTION = [0.25, 0.50, 0.75]
LLAVA_ID  = "llava-hf/llava-1.5-7b-hf"
CLIP_ID   = "openai/clip-vit-large-patch14"   # must match LLaVA's vision tower
MAX_TOKENS = 32   # sufficient for most VQA answers

In [ ]:
# ── GQA ──────────────────────────────────────────────────────────────────────
gqa_results = run_llava_benchmark(
    dataset_csv="../data/gqa_val.csv",
    output_dir="../outputs/gqa_eval",
    retention_list=RETENTION,
    llava_model_id=LLAVA_ID,
    clip_model_id=CLIP_ID,
    device=DEVICE,
    max_new_tokens=MAX_TOKENS,
)
print(gqa_results.groupby("retention_ratio")[["vqa_acc_papit","vqa_acc_random","vqa_acc_unpruned"]].mean())

In [ ]:
# ── VQA v2 ───────────────────────────────────────────────────────────────────
vqa_results = run_llava_benchmark(
    dataset_csv="../data/vqa_v2_val.csv",
    output_dir="../outputs/vqa_v2_eval",
    retention_list=RETENTION,
    llava_model_id=LLAVA_ID,
    clip_model_id=CLIP_ID,
    device=DEVICE,
    max_new_tokens=MAX_TOKENS,
)
print(vqa_results.groupby("retention_ratio")[["vqa_acc_papit","vqa_acc_random","vqa_acc_unpruned"]].mean())

In [ ]:
# ── TextVQA ───────────────────────────────────────────────────────────────────
tvqa_results = run_llava_benchmark(
    dataset_csv="../data/textvqa_val.csv",
    output_dir="../outputs/textvqa_eval",
    retention_list=RETENTION,
    llava_model_id=LLAVA_ID,
    clip_model_id=CLIP_ID,
    device=DEVICE,
    max_new_tokens=MAX_TOKENS,
)
print(tvqa_results.groupby("retention_ratio")[["vqa_acc_papit","vqa_acc_random","vqa_acc_unpruned","papit_patch_recall"]].mean())

---
## Section 3 — Paper Figures & Table

Reads the summary CSVs written by Section 2 and produces:
- **Figure 1** — Accuracy–Efficiency Pareto curve (3 subplots)
- **Figure 2** — TextVQA patch recall vs retention ratio
- **Table 1** — Main results (saved as CSV for LaTeX)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

Path("../outputs").mkdir(exist_ok=True)

DATASETS = [
    ("GQA",     "../outputs/gqa_eval/llava_benchmark_summary.csv"),
    ("VQA v2",  "../outputs/vqa_v2_eval/llava_benchmark_summary.csv"),
    ("TextVQA", "../outputs/textvqa_eval/llava_benchmark_summary.csv"),
]

summaries = {name: pd.read_csv(path) for name, path in DATASETS}

In [ ]:
# Figure 1 — Accuracy–Efficiency Pareto Curve
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, df) in zip(axes, summaries.items()):
    ax.plot(df["avg_relative_flops"], df["avg_vqa_acc_papit"],  "o-",  label="PAPIT",    color="steelblue")
    ax.plot(df["avg_relative_flops"], df["avg_vqa_acc_random"], "s--", label="Random",   color="coral")
    ax.axhline(df["avg_vqa_acc_unpruned"].iloc[0], color="gray", linestyle=":", label="Unpruned")

    # Annotate retention ratios
    for _, row in df.iterrows():
        ax.annotate(f"k={int(row['retention_ratio']*100)}%",
                    (row["avg_relative_flops"], row["avg_vqa_acc_papit"]),
                    textcoords="offset points", xytext=(5, 4), fontsize=8)

    ax.set_title(name, fontsize=13)
    ax.set_xlabel("Relative Attention FLOPs")
    ax.set_ylabel("VQA Soft Accuracy")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle("PAPIT: Accuracy–Efficiency Pareto Curve", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("../outputs/pareto_curve.pdf", bbox_inches="tight")
plt.savefig("../outputs/pareto_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/pareto_curve.pdf")

In [ ]:
# Figure 2 — TextVQA Patch Recall vs Retention Ratio
df = summaries["TextVQA"]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["retention_ratio"], df["avg_papit_patch_recall"],  "o-",  label="PAPIT",  color="steelblue")
ax.plot(df["retention_ratio"], df["avg_random_patch_recall"], "s--", label="Random", color="coral")
ax.set_xlabel("Retention Ratio (k / N)")
ax.set_ylabel("Text-Region Patch Recall")
ax.set_title("TextVQA: Patch Recall on Text Regions")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../outputs/textvqa_patch_recall.pdf", bbox_inches="tight")
plt.savefig("../outputs/textvqa_patch_recall.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/textvqa_patch_recall.pdf")

In [ ]:
# Table 1 — Main Results
rows = []
for name, df in summaries.items():
    for _, row in df.iterrows():
        rows.append({
            "Dataset":   name,
            "k":         f"{int(row['retention_ratio']*100)}%",
            "Unpruned":  f"{row['avg_vqa_acc_unpruned']:.3f}",
            "Random":    f"{row['avg_vqa_acc_random']:.3f}",
            "PAPIT":     f"{row['avg_vqa_acc_papit']:.3f}",
            "Rel. FLOPs":f"{row['avg_relative_flops']:.3f}",
            "Latency (s)":f"{row['avg_latency_papit_sec']:.2f}",
        })

table_df = pd.DataFrame(rows)
table_df.to_csv("../outputs/table1.csv", index=False)
print(table_df.to_string(index=False))

---
## Section 4 — (Optional) Risk Awareness Ablation

Compares Base vs Risk-aware pruning on a small set of adversarial images
(images with injected instruction text or safety-critical patches).

Prepare a folder `data/adversarial/` with images and a matching `adversarial.csv`
(columns: `image_path`, `question`, `answer_list`).

In [ ]:
from papit.benchmark.efficiency import measure_variant
import pandas as pd

ADVERSARIAL_CSV = "../data/adversarial/adversarial.csv"   # edit path as needed

adv_df = pd.read_csv(ADVERSARIAL_CSV)
rows = []
for _, sample in adv_df.iterrows():
    for risk in [False, True]:
        r = measure_variant(
            image_path=sample["image_path"],
            question=sample["question"],
            retention_ratio=0.5,
            risk_aware=risk,
            runs=1,
            device=DEVICE,
        )
        r["image"] = sample["image_path"]
        r["question"] = sample["question"]
        rows.append(r)

risk_df = pd.DataFrame(rows)
risk_df.to_csv("../outputs/risk_ablation.csv", index=False)
print(risk_df[["image", "risk_aware", "avg_tokens_kept", "avg_prune_latency_sec"]].to_string(index=False))